### Cleaning / catalog building.

>Inputs : 

* processed/recipes_flagged
* data/pp_recipes.csv (for ingredient quantities + directions)

> Outputs: 

* processed/catalog.parquet  [6,525 × 49]         (~N_CATALOG recipes, full text + nutrition + percentiles + constraint status)

* processed/recipe_lookup.parquet  [243,115 × 19]   (all quality_ok reviewed recipes, slim; used to build historical profiles)

* processed/baselines.json    (review-weighted base rates used for 'is this tendency real?' tests)

* processed/catalog_report.json

In [1]:
import json
import re
import ast
from collections import Counter

import numpy as np
import pandas as pd

from config import (DATA_DIR, PROC_DIR, SEED, read_table, write_table, parse_list_cols, informative_tags,
                    norm_ingredient, CONSTRAINT_COL, CONSTRAINT_TAG)

In [2]:
N_CATALOG = 6000
MIN_PER_STRATUM = 150
MIN_TAG_RECIPES = 50          # tags / ingredients rarer than this get no baseline entry
MIN_ING_RECIPES = 100

In [3]:
PCT_SOURCES = {
    "calories_pct": "calories [cal]", "sodium_pct": "sodium [mg]", "sugars_pct": "sugars [g]",
    "satfat_pct": "saturatedFat [g]", "protein_pct": "protein [g]", "fiber_pct": "dietaryFiber [g]",
    "who_pct": "who_score", "duration_pct": "duration",
}

In [4]:
def wstats(x, w):
    m = np.average(x, weights=w)
    return float(m), float(np.sqrt(np.average((x - m) ** 2, weights=w)))


def format_ingredients(s):
    try:
        d = ast.literal_eval(s)
    except Exception:
        return []
    out = []
    if isinstance(d, dict):
        for items in d.values():
            for it in items:
                qty = re.sub(r"\s*time\(s\)\s*", " ", str(it[1])).strip() if len(it) > 1 else ""
                out.append(re.sub(r"\s+", " ", f"{qty} {it[0]}").strip())
    return out

def format_directions(s):
    try:
        v = ast.literal_eval(s)
    except Exception:
        return str(s) if isinstance(s, str) else ""
    if isinstance(v, dict):
        v = [x for vals in v.values() for x in (vals if isinstance(vals, list) else [vals])]
    return " ".join(f"{i}. {str(x).strip()}" for i, x in enumerate(v, 1)) if isinstance(v, (list, tuple)) else str(v)

In [ ]:
rng = np.random.default_rng(SEED)
print("loading recipes_flagged ...")
rf = read_table("recipes_flagged")
Q = rf[rf["quality_ok"].astype(bool)].copy()
del rf
print(f"quality_ok recipes: {len(Q):,}")
parse_list_cols(Q)

# ---- derived columns
Q["tags_inf"] = Q["tags_list"].apply(informative_tags)
Q["tags_str"] = Q["tags_inf"].apply("|".join)
Q["ing_norm"] = Q["ingredient_names"].apply(lambda l: sorted({n for n in map(norm_ingredient, l) if n}))
Q["ing_str"] = Q["ing_norm"].apply("|".join)
Q["ing_text"] = Q["ingredient_names"].apply(lambda l: " ; ".join(l).lower())
for dst, src in PCT_SOURCES.items():
    Q[dst] = Q[src].rank(pct=True)
Q["nutri_level"] = (Q["nutri_score"] * 4).round().astype(int)      # 0..4, higher = healthier (verified in EDA 9.4)
Q["fsa_level"] = (Q["fsa_score"] * 8).round().astype(int)          # 0..8, higher = healthier
for c, rule in CONSTRAINT_COL.items():
    Q[rule] = Q[rule].astype(bool)
for c, tag in CONSTRAINT_TAG.items():
    has_tag = Q["tags_list"].apply(lambda l, t=tag: t in l)
    Q[f"{c}_status"] = np.where(~Q[CONSTRAINT_COL[c]], "violates", np.where(has_tag, "verified", "rule_only"))

# ---- review-weighted baselines (what a *typical reviewed* recipe looks like)
Qr = Q[Q["n_reviews"] > 0]
w = Qr["n_reviews"].to_numpy(float)
base = {"pct_metrics": {}, "rule_rates": {}, "tag_rates": {}, "ingredient_rates": {}}
for dst in PCT_SOURCES:
    m, sd = wstats(Qr[dst].to_numpy(float), w)
    base["pct_metrics"][dst] = {"mean": m, "sd": sd}
for rule in CONSTRAINT_COL.values():
    base["rule_rates"][rule] = float(np.average(Qr[rule].to_numpy(float), weights=w))

def weighted_rates(col, min_recipes):
    sub = Qr[Qr[col].apply(len) > 0][[col, "n_reviews"]]
    total = sub["n_reviews"].sum()
    ex = sub.explode(col)
    g = ex.groupby(col).agg(w=("n_reviews", "sum"), n=("n_reviews", "size"))
    g = g[g["n"] >= min_recipes]
    return {k: float(v / total) for k, v in g["w"].items()}

base["tag_rates"] = weighted_rates("tags_inf", MIN_TAG_RECIPES)
base["ingredient_rates"] = weighted_rates("ing_norm", MIN_ING_RECIPES)
with open(PROC_DIR / "baselines.json", "w") as f:
    json.dump(base, f)
print(f"baselines: {len(base['tag_rates'])} tags, {len(base['ingredient_rates'])} ingredients")

# ---- slim lookup for historical profiling
lookup_cols = ["new_recipe_id", "duration", "nutri_level", "fsa_level", "tags_str", "ing_str"] + list(PCT_SOURCES) + list(CONSTRAINT_COL.values())
write_table(Qr[lookup_cols], "recipe_lookup")

# ---- catalog: candidate recipes, stratified sample
C = Q[Q["catalog_candidate"].astype(bool)].copy()
C["time_bucket"] = pd.cut(C["duration"], [0, 30, 60, 1e9], labels=["<=30", "31-60", ">60"])
C["stratum"] = C["nutri_level"].astype(str) + "|" + C["time_bucket"].astype(str)
sizes = C["stratum"].value_counts()
target = {s: int(min(n, max(MIN_PER_STRATUM, round(N_CATALOG * n / len(C))))) for s, n in sizes.items()}
picks = [g.sample(target[s], random_state=SEED) for s, g in C.groupby("stratum")]
cat = pd.concat(picks).reset_index(drop=True)
print(f"candidates={len(C):,} -> catalog={len(cat):,} recipes over {len(sizes)} strata")

# ---- attach ingredient quantities + directions from the raw file (catalog ids only)
ids = set(cat["new_recipe_id"])
parts = []
for ch in pd.read_csv(DATA_DIR / "pp_recipes.csv", usecols=["new_recipe_id", "ingredients", "directions"], chunksize=50_000):
    parts.append(ch[ch["new_recipe_id"].isin(ids)])
raw = pd.concat(parts)
raw["ingredients_display"] = raw["ingredients"].apply(format_ingredients)
raw["directions_text"] = raw["directions"].apply(format_directions)
cat = cat.merge(raw[["new_recipe_id", "ingredients_display", "directions_text"]], on="new_recipe_id", how="left")

keep = (["new_recipe_id", "title", "description", "duration", "servingsPerRecipe", "servingSize [g]",
            "calories [cal]", "totalFat [g]", "saturatedFat [g]", "cholesterol [mg]", "sodium [mg]",
            "totalCarbohydrate [g]", "dietaryFiber [g]", "sugars [g]", "protein [g]",
            "who_score", "fsa_score", "nutri_score", "nutri_level", "fsa_level", "n_reviews", "mean_true_rating",
            "tags_str", "ing_text", "ingredient_names", "ingredients_display", "directions_text", "stratum"]
            + list(PCT_SOURCES) + list(CONSTRAINT_COL.values()) + [f"{c}_status" for c in CONSTRAINT_TAG]
            + [c for c in cat.columns if c.startswith("under_")])
cat = cat[keep]
write_table(cat, "catalog")

report = {
        "quality_ok_recipes": int(len(Q)), "candidate_recipes": int(len(C)), "catalog_size": int(len(cat)),
        "strata": cat["stratum"].value_counts().to_dict(),
        "constraint_share_pct": {k: round(float(cat[v].mean() * 100), 1) for k, v in CONSTRAINT_COL.items()},
        "status_counts": {c: cat[f"{c}_status"].value_counts().to_dict() for c in CONSTRAINT_TAG},
        "under_30min_pct": round(float(cat["under_30min"].mean() * 100), 1),
        "missing_directions": int((cat["directions_text"].fillna("") == "").sum()),
    }
    
with open(PROC_DIR / "catalog_report.json", "w") as f:
    json.dump(report, f, indent=2)
print(json.dumps(report, indent=2))

loading recipes_flagged ...
quality_ok recipes: 448,652
baselines: 420 tags, 1947 ingredients
saved processed\recipe_lookup.parquet  (243115, 19)
candidates=102,317 -> catalog=6,525 recipes over 15 strata
saved processed\catalog.parquet  (6525, 49)
{
  "quality_ok_recipes": 448652,
  "candidate_recipes": 102317,
  "catalog_size": 6525,
  "strata": {
    "1|<=30": 994,
    "0|31-60": 787,
    "1|31-60": 749,
    "0|>60": 727,
    "0|<=30": 711,
    "1|>60": 619,
    "2|<=30": 515,
    "2|31-60": 293,
    "2|>60": 200,
    "3|<=30": 180,
    "3|31-60": 150,
    "3|>60": 150,
    "4|31-60": 150,
    "4|<=30": 150,
    "4|>60": 150
  },
  "constraint_share_pct": {
    "vegetarian": 60.7,
    "vegan": 18.6,
    "dairy-free": 41.9,
    "gluten-free": 52.8,
    "nut-free": 86.9
  },
  "status_counts": {
    "vegetarian": {
      "rule_only": 2950,
      "violates": 2567,
      "verified": 1008
    },
    "vegan": {
      "violates": 5312,
      "rule_only": 920,
      "verified": 293
    },
 

In [11]:
import pyarrow.parquet as pq

pd.set_option("display.max_columns", None)

# Open the file structure and stream just the first 2 rows
recipe_lookup = pq.read_table(PROC_DIR / "recipe_lookup.parquet")
catalog = pq.read_table(PROC_DIR / "catalog.parquet")

recipe_lookup_two_rows = recipe_lookup.slice(0, 2).to_pandas()
catalog_two_rows = catalog.slice(0, 2).to_pandas()

In [ ]:
recipe_lookup_two_rows # (243,115 × 19)

,new_recipe_id,duration,nutri_level,fsa_level,tags_str,ing_str,calories_pct,sodium_pct,sugars_pct,satfat_pct,protein_pct,fiber_pct,who_pct,duration_pct,rule_vegetarian,rule_vegan,rule_dairy_free,rule_gluten_free,rule_nut_free
0,1,20.0,1,5,north-american|for-1-or-2|very-low-carbs|appet...,avocado|cilantro|garlic salt|jalapeno|large to...,0.648212,0.615578,0.494374,0.362908,0.945058,0.917714,0.978451,0.192863,False,False,True,True,True
1,3,375.0,0,3,very-low-carbs|main-dish|lamb-sheep|greek|euro...,dried mint flakes|dried oregano|dried rosemary...,0.792386,0.276981,0.334265,0.853752,0.944419,0.461492,0.960241,0.975772,False,False,True,True,True


In [ ]:
catalog_two_rows # (6,525 × 49)

,new_recipe_id,title,description,duration,servingsPerRecipe,servingSize [g],calories [cal],totalFat [g],saturatedFat [g],cholesterol [mg],sodium [mg],totalCarbohydrate [g],dietaryFiber [g],sugars [g],protein [g],who_score,fsa_score,nutri_score,nutri_level,fsa_level,n_reviews,mean_true_rating,tags_str,ing_text,ingredient_names,ingredients_display,directions_text,stratum,calories_pct,sodium_pct,sugars_pct,satfat_pct,protein_pct,fiber_pct,who_pct,duration_pct,rule_vegetarian,rule_vegan,rule_dairy_free,rule_gluten_free,rule_nut_free,vegetarian_status,vegan_status,dairy-free_status,gluten-free_status,under_15min,under_30min,under_45min,under_60min
0,490599,Mandarin Grilled Pork Steaks,"The first time these steaks hit my grill, one ...",40.0,4,113.0,241.9,6.7,2.3,66.7,1793.1,22.1,0.2,21.0,23.5,0.203961,0.375,0.0,0,3,4,4.500000,healthy|pork|barbecue|meat|grilling,beef bouillon cube ; boiling water ; ground gi...,"[beef bouillon cube, boiling water, ground gin...","[1 beef bouillon cube, 0.3333333333333333 cup ...","1. MARINADE: In small mixing bowl, stir bouill...",0|31-60,0.359920,0.935372,0.788508,0.310828,0.719505,0.075019,0.356625,0.490776,False,False,True,False,True,violates,violates,rule_only,violates,False,False,True,True
1,83565,Lemon Chicken Spaghetti Toss,I found this recipe in Quick Cooking a few yea...,35.0,4,268.0,504.2,18.1,5.7,124.2,257.3,36.7,0.7,0.8,46.7,0.274609,0.500,0.0,0,4,47,4.585106,main-dish|pasta|poultry|chicken|meat|pasta-ric...,"green onions, chopped ; cloves garlic, minced ...","[green onions, chopped, cloves garlic, minced,...","[5 green onions, chopped, 2 cloves garlic, min...","1. In a large skillet, saute onions& garlic in...",0|31-60,0.730375,0.394065,0.090102,0.545509,0.927759,0.198507,0.816314,0.426474,False,False,False,False,True,violates,violates,violates,violates,False,False,True,True
